In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import plotly.express as px

In [2]:
df = pd.read_csv('spotify-2023.csv', encoding='ISO-8859-1')

df['track_name'] = df['track_name'].str.encode('latin1').str.decode('utf-8', errors='ignore')

df['streams'] = pd.to_numeric(df['streams'].astype(str).str.replace(',', ''), errors='coerce')

# Optional: Drop any rows where 'streams' couldn't be converted
df = df.dropna(subset=['streams'])

# Convert to integer if you prefer
df['streams'] = df['streams'].astype('int64')


FileNotFoundError: [Errno 2] No such file or directory: 'spotify-2023.csv'

In [ ]:
df

,track_name,artist(s)_name,artist_count,released_year,released_month,released_day,in_spotify_playlists,in_spotify_charts,streams,in_apple_playlists,...,bpm,key,mode,danceability_%,valence_%,energy_%,acousticness_%,instrumentalness_%,liveness_%,speechiness_%
0,Seven (feat. Latto) (Explicit Ver.),"Latto, Jung Kook",2,2023,7,14,553,147,141381703,43,...,125,B,Major,80,89,83,31,0,8,4
1,LALA,Myke Towers,1,2023,3,23,1474,48,133716286,48,...,92,C#,Major,71,61,74,7,0,10,4
2,vampire,Olivia Rodrigo,1,2023,6,30,1397,113,140003974,94,...,138,F,Major,51,32,53,17,0,31,6
3,Cruel Summer,Taylor Swift,1,2019,8,23,7858,100,800840817,116,...,170,A,Major,55,58,72,11,0,11,15
4,WHERE SHE GOES,Bad Bunny,1,2023,5,18,3133,50,303236322,84,...,144,A,Minor,65,23,80,14,63,11,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
948,My Mind & Me,Selena Gomez,1,2022,11,3,953,0,91473363,61,...,144,A,Major,60,24,39,57,0,8,3
949,Bigger Than The Whole Sky,Taylor Swift,1,2022,10,21,1180,0,121871870,4,...,166,F#,Major,42,7,24,83,1,12,6
950,A Veces (feat. Feid),"Feid, Paulo Londra",2,2022,11,3,573,0,73513683,2,...,92,C#,Major,80,81,67,4,0,8,6
951,En La De Ella,"Feid, Sech, Jhayco",3,2022,10,20,1320,0,133895612,29,...,97,C#,Major,82,67,77,8,0,12,5


In [ ]:
print(df.nlargest(10, 'streams')[['track_name', 'streams']])


                                        track_name     streams
55                                 Blinding Lights  3703895074
179                                   Shape of You  3562543890
86                               Someone You Loved  2887241814
620                                   Dance Monkey  2864791672
41   Sunflower - Spider-Man: Into the Spider-Verse  2808096550
162                                      One Dance  2713922350
84                       STAY (with Justin Bieber)  2665343922
140                                       Believer  2594040133
725                                         Closer  2591224264
48                                         Starboy  2565529693


# 1 Top 10 Most Streamed Songs By Track Name(bar chart)

In [ ]:

top_10_tracks = df.sort_values(by='streams', ascending=False).head(10)

fig = px.bar(
    top_10_tracks,
    x='track_name',
    y='streams',
    color='artist(s)_name',
    title='Top 10 Most Streamed Songs by Track Name',
    labels={'artist(s)_name': 'Artist', 'streams': 'Number of Streams'},
    height=500,
    width=900
)

fig.update_layout(
    xaxis_tickangle=-45,
    margin=dict(t=50, b=150)
)

fig.show()


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

# 2 Top 10 Most Streamed Artists(bar chart)

In [ ]:
top_10_artists = df.groupby('artist(s)_name')['streams'].sum().sort_values(ascending=False).head(10).reset_index()

fig = px.bar(
    top_10_artists,
    x='artist(s)_name',
    y='streams',
    title='Top 10 Most Streamed Artists',
    labels={'artist(s)_name': 'Artist', 'streams': 'Total Streams'},
    height=500,
    width=900,
    color='artist(s)_name'
)

fig.update_layout(
    showlegend=False,
    xaxis_tickangle=-45,
    margin=dict(t=50, b=150)
)

fig.show()


# 3 Top 30 Most Streamed Artists(pie chart)

In [ ]:
top_30_artists = df.groupby('artist(s)_name')['streams'].sum().sort_values(ascending=False).head(30).reset_index()

fig = px.pie(
    top_30_artists,
    names='artist(s)_name',
    values='streams',
    title='Top 10 Most Streamed Artists',
    color_discrete_sequence=px.colors.sequential.RdBu
)

fig.update_traces(textposition='inside', textinfo='percent+label')

fig.show()


# 4 Top 10 in_spotify_playlists by Total Streams(treemap)

In [ ]:
top_artists = df.groupby('in_spotify_playlists')['streams'].sum().nlargest(10).reset_index()

fig = px.treemap(
    top_artists,
    path=['in_spotify_playlists'],
    values='streams',
    title='Top 10 in_spotify_playlists by Total Streams',
    color='streams',
    color_continuous_scale='Blues'
)
fig.show()


# 5 Streams by Release Year(line chart)

In [ ]:
streams_by_year = df.groupby('released_year')['streams'].sum().reset_index()

fig = px.line(
    streams_by_year,
    x='released_year',
    y='streams',
    title='Streams by Release Year',
    markers=True,
    labels={'released_year': 'Year', 'streams': 'Total Streams'}
)
fig.show()


# 6 Energy vs Danceability(bubble scatter plot)

In [ ]:
fig = px.scatter(
    df,
    x='energy_%',
    y='danceability_%',
    size='streams',
    color='artist(s)_name',
    title='Energy vs Danceability (Bubble Size = Streams)',
    labels={'energy_%': 'Energy', 'danceability_%': 'Danceability'}
)
fig.show()


# 7 Distribution of BPM in Tracks(histogram)

In [ ]:
fig = px.histogram(
    df,
    x='bpm',
    nbins=30,
    title='Distribution of BPM in Tracks',
    labels={'bpm': 'Beats Per Minute (BPM)'}
)
fig.show()


# 8 Danceability Distribution by Mode(box chart)

In [ ]:
fig = px.box(
    df,
    x='mode',
    y='danceability_%',
    title='Danceability Distribution by Mode (0 = Minor, 1 = Major)',
    labels={'mode': 'Mode', 'danceability_%': 'Danceability'}
)
fig.show()


# 9 Streams by Year → Artist → Track(sunburst chart)

In [ ]:
top_years = df['released_year'].value_counts().nlargest(3).index
filtered = df[df['released_year'].isin(top_years)]

fig = px.sunburst(
    filtered,
    path=['released_year', 'artist(s)_name', 'track_name'],
    values='streams',
    title='Streams by Year → Artist → Track'
)
fig.show()


# 10 Streams by Release Year(violin plot)

In [ ]:
fig = px.violin(
    df,
    x='released_year',
    y='streams',
    color='released_year',  # add this line for color
    title='Violin Plot of Streams by Release Year',
    labels={'released_year': 'Year', 'streams': 'Streams'},
    box=True,
    points='all'
)

fig.show()

# "python -m streamlit run app.py" is used in the terminal for running streamlit projects